# Notebook 06 — LSTM

In [1]:
# ==============================================================================
# PARTE 1: CONFIGURACIÓN, LIBRERÍAS Y SEMILLAS
# ==============================================================================
import os
import random
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Configuración visual de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

# 1. Fijar Semillas para asegurar reproducibilidad
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. Configuración de Hiperparámetros Globales
FAMILIAS = ['106', '124', '233']  # Familias de ventas objetivo
COLUMNAS_EXOG = ['BRENT_USD', 'BCE_%']  # Regresores exógenos

LOOKBACK = 12   # Usamos los últimos 12 meses para predecir...
FORECAST = 1    # ...el siguiente mes (t+1)
TEST_SIZE = 12  # Reservamos los últimos 12 meses como test/validación

print("Librerías importadas y semillas configuradas con éxito.")

Librerías importadas y semillas configuradas con éxito.


In [2]:
# ==============================================================================
# PARTE 2: CARGA Y PROCESAMIENTO DE VENTAS MENSUALES
# ==============================================================================
print("[Proceso] Cargando datos de ventas...")

ruta_archivo = '../data/processed/datos.xlsx'

if not os.path.exists(ruta_archivo):
    raise FileNotFoundError(
        f"No se encontró '{ruta_archivo}'. Verifica que ejecutas el notebook "
        f"desde la carpeta 'notebooks/' del proyecto (misma ubicación relativa "
        f"que 05_XGBoost_LightGBM.ipynb y 06_LTSM.ipynb)."
    )

df_raw = pd.read_excel(ruta_archivo)
df_raw['FECHA'] = pd.to_datetime(df_raw['FECHA'])

df_ventas = df_raw[df_raw['TIPO_OPERACION'] == 'VENTA'].copy()
df_ventas['PERIODO'] = df_ventas['FECHA'].dt.to_period('M').dt.to_timestamp()
df_ventas['FAMILIA'] = df_ventas['FAMILIA'].astype(str)   # alinea con FAMILIAS = ['106','124','233']

# Pivotar para tener columnas por familia y filas por mes
ventas_mensual = (
    df_ventas
    .groupby(['PERIODO', 'FAMILIA'])['PESO_BRUTO']
    .sum()
    .unstack('FAMILIA')
    .fillna(0)
)
ventas_mensual = ventas_mensual[FAMILIAS]

# Remuestrear a frecuencia mensual explícita (Month Start)
ventas_mensual = ventas_mensual.resample('MS').sum()

print(f"Dataset de ventas listo. Rango temporal: {ventas_mensual.index.min().date()} a {ventas_mensual.index.max().date()}")
print(ventas_mensual.head())

[Proceso] Cargando datos de ventas...
Dataset de ventas listo. Rango temporal: 2010-12-01 a 2026-02-01
FAMILIA     106  124         233
PERIODO                         
2010-12-01  0.0  0.0  61578.7102
2011-01-01  0.0  0.0      0.0000
2011-02-01  0.0  0.0      0.0000
2011-03-01  0.0  0.0      0.0000
2011-04-01  0.0  0.0   1550.1600


In [3]:
# ==============================================================================
# PARTE 3: REGRESORES EXÓGENOS (BRENT + BCE REAL)
# ==============================================================================
print("[Proceso] Descargando e integrando variables macroeconómicas exógenas...")

fecha_inicio = ventas_mensual.index.min().strftime('%Y-%m-%d')
fecha_fin = (ventas_mensual.index.max() + pd.DateOffset(months=1)).strftime('%Y-%m-%d')

# 1. Brent (igual que antes)
try:
    brent_daily = yf.download('BZ=F', start=fecha_inicio, end=fecha_fin, progress=False)
    if isinstance(brent_daily.columns, pd.MultiIndex):
        brent_daily = brent_daily['Close'].iloc[:, 0]
    else:
        brent_daily = brent_daily['Close']
    brent_monthly = brent_daily.resample('MS').mean().ffill()
except Exception as e:
    print(f"[Error] No se pudo descargar desde Yahoo Finance: {e}. Usando datos simulados.")
    brent_monthly = pd.Series(np.random.uniform(70, 95, size=len(ventas_mensual)), index=ventas_mensual.index)

# 2. BCE real (serie oficial hardcodeada, igual que en 06_LTSM.ipynb)
# Fuente: https://www.ecb.europa.eu/stats/policy_and_exchange_rates/key_ecb_interest_rates
bce_cambios = {
    '2009-01': 2.00, '2009-03': 1.00, '2009-04': 0.75,
    '2009-05': 0.50, '2009-06': 0.25,
    '2011-04': 0.50, '2011-07': 0.75,
    '2012-07': 0.00,
    '2014-06': -0.10, '2015-12': -0.30, '2016-03': -0.40,
    '2019-09': -0.50,
    '2022-07':  0.00, '2022-09':  0.75,
    '2022-11':  1.50, '2022-12':  2.00,
    '2023-02':  2.50, '2023-03':  3.00,
    '2023-05':  3.25, '2023-06':  3.50,
    '2023-09':  4.00,
    '2024-06':  3.75, '2024-09':  3.50,
    '2024-10':  3.25, '2024-12':  3.00,
    '2025-01':  2.75, '2025-03':  2.50,
    '2025-04':  2.25,
}
idx_bce = pd.period_range(ventas_mensual.index.min(), ventas_mensual.index.max(), freq='M')
tipo_actual = 2.00
valores_bce = []
for p in idx_bce:
    if str(p) in bce_cambios:
        tipo_actual = bce_cambios[str(p)]
    valores_bce.append(tipo_actual)
bce_monthly = pd.Series(valores_bce, index=idx_bce.to_timestamp())

# 3. Unificación
df_exog = pd.DataFrame({
    'BRENT_USD': brent_monthly,
    'BCE_%': bce_monthly
}, index=ventas_mensual.index).ffill().bfill()

print("Variables exógenas unificadas (BCE real):")
print(df_exog.head())

[Proceso] Descargando e integrando variables macroeconómicas exógenas...
Variables exógenas unificadas (BCE real):
             BRENT_USD  BCE_%
PERIODO                      
2010-12-01   92.185908    2.0
2011-01-01   96.786316    2.0
2011-02-01  103.942105    2.0
2011-03-01  114.649545    2.0
2011-04-01  123.050526    0.5


In [4]:
# ==============================================================================
# PARTE 4: UNIFICACIÓN, WINSORIZACIÓN, LOG-DIFF Y SEPARACIÓN TRAIN/VAL/TEST
# ==============================================================================
print("[Proceso] Unificando series, winsorizando y aplicando transformaciones...")

VAL_SIZE  = 12   # tramo de validación real, para early stopping
TEST_SIZE = 12   # tramo de test final — nunca visto durante el entrenamiento

df_unificado = ventas_mensual.join(df_exog[COLUMNAS_EXOG], how='inner')

# 1. Winsorización SOLO con estadísticos de train (evita fuga de datos)
n_total_pre = len(df_unificado)
n_train_pre = n_total_pre - VAL_SIZE - TEST_SIZE

df_wins = df_unificado.copy()
for col in FAMILIAS:
    p95_train = df_unificado[col].iloc[:n_train_pre].quantile(0.95)
    df_wins[col] = df_unificado[col].clip(upper=p95_train)
    print(f"  [{col}] winsorización train al p95 = {p95_train:,.0f}")

# 2. Log1p sobre ventas (estabiliza varianza)
df_transformed = df_wins.copy()
for col in FAMILIAS:
    df_transformed[col] = np.log1p(df_transformed[col])

# 3. Diferenciación — revisa tus tests ADF/KPSS del Notebook 02 antes de aplicarla
#    a las exógenas; aquí se asume que ventas y Brent la necesitan y BCE se deja
#    en nivel (ya es casi constante por tramos). Ajusta según tus resultados reales.
df_diff = df_transformed.copy()
for col in FAMILIAS + ['BRENT_USD']:
    df_diff[col] = df_diff[col].diff()

# 4. Lags exógenos
columnas_exog_lag = []
for col in COLUMNAS_EXOG:
    nombre_lag = f'{col}_LAG1'
    df_diff[nombre_lag] = df_diff[col].shift(1)
    columnas_exog_lag.append(nombre_lag)

df_diff = df_diff.dropna()
columnas_ventas_str = [str(c) for c in FAMILIAS]
columnas_exog_str   = columnas_exog_lag
df_diff = df_diff[columnas_ventas_str + columnas_exog_str]

# 5. Split cronológico estricto en TRES tramos (recalculado tras el dropna)
n_total = len(df_diff)
n_train = n_total - VAL_SIZE - TEST_SIZE

df_train_raw = df_diff.iloc[:n_train].copy()
df_val_raw   = df_diff.iloc[n_train:n_train + VAL_SIZE].copy()
df_test_raw  = df_diff.iloc[n_train + VAL_SIZE:].copy()

print(f"Train: {len(df_train_raw)} meses | Val: {len(df_val_raw)} meses | Test: {len(df_test_raw)} meses")

# 6. Escalado ajustado SOLO en train
scaler = MinMaxScaler(feature_range=(-1, 1))
df_train_scaled = pd.DataFrame(scaler.fit_transform(df_train_raw), columns=df_train_raw.columns, index=df_train_raw.index)
df_val_scaled   = pd.DataFrame(scaler.transform(df_val_raw),       columns=df_val_raw.columns,   index=df_val_raw.index)
df_test_scaled  = pd.DataFrame(scaler.transform(df_test_raw),      columns=df_test_raw.columns,  index=df_test_raw.index)

[Proceso] Unificando series, winsorizando y aplicando transformaciones...
  [106] winsorización train al p95 = 23,589
  [124] winsorización train al p95 = 57,710
  [233] winsorización train al p95 = 108,026
Train: 157 meses | Val: 12 meses | Test: 12 meses


In [5]:
# ==============================================================================
# PARTE 5: VENTANAS TEMPORALES POR FAMILIA (MODELOS INDEPENDIENTES)
# ==============================================================================
def preparar_ventanas(df, lookback, forecast, target_col, exog_cols):
    total_cols = [target_col] + list(exog_cols)
    data_array = df[total_cols].values
    X, y = [], []
    for i in range(len(df) - lookback - forecast + 1):
        X.append(data_array[i : i + lookback])
        y.append(data_array[i + lookback + forecast - 1, 0])
    return np.array(X), np.array(y)

datos_por_familia = {}

for familia in FAMILIAS:
    X_tr, y_tr = preparar_ventanas(df_train_scaled, LOOKBACK, FORECAST, familia, columnas_exog_str)

    # Para val/test, anteponemos las últimas LOOKBACK filas del tramo anterior
    # (solo como features de contexto, nunca como etiquetas) para no perder ventanas
    df_val_ctx  = pd.concat([df_train_scaled.iloc[-LOOKBACK:], df_val_scaled])
    df_test_ctx = pd.concat([df_val_scaled.iloc[-LOOKBACK:],   df_test_scaled])

    X_va, y_va = preparar_ventanas(df_val_ctx,  LOOKBACK, FORECAST, familia, columnas_exog_str)
    X_te, y_te = preparar_ventanas(df_test_ctx, LOOKBACK, FORECAST, familia, columnas_exog_str)

    datos_por_familia[familia] = dict(X_tr=X_tr, y_tr=y_tr, X_va=X_va, y_va=y_va, X_te=X_te, y_te=y_te)
    print(f"[{familia}] X_tr={X_tr.shape}  X_va={X_va.shape}  X_te={X_te.shape}")

[106] X_tr=(145, 12, 3)  X_va=(12, 12, 3)  X_te=(12, 12, 3)
[124] X_tr=(145, 12, 3)  X_va=(12, 12, 3)  X_te=(12, 12, 3)
[233] X_tr=(145, 12, 3)  X_va=(12, 12, 3)  X_te=(12, 12, 3)


In [6]:
# ==============================================================================
# PARTE 6: MODELOS LSTM INDEPENDIENTES POR FAMILIA (ARQUITECTURA REDUCIDA)
# ==============================================================================
from tensorflow.keras import regularizers

CONFIG_POR_FAMILIA = {
    '106': {'units': 16, 'dropout': 0.4, 'lr': 5e-4},   # intermitente -> más regularización
    '124': {'units': 32, 'dropout': 0.4, 'lr': 3e-4},   # heterocedástica -> algo más de capacidad
    '233': {'units': 16, 'dropout': 0.3, 'lr': 5e-4},   # regular -> modelo simple, evitar sobreajuste
}

def build_model(lookback, n_features, units, dropout, lr):
    reg = regularizers.l2(1e-2)
    model = models.Sequential([
        layers.Input(shape=(lookback, n_features)),
        layers.LSTM(units, return_sequences=False,
                    kernel_regularizer=reg, recurrent_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(max(units // 2, 4), activation='relu', kernel_regularizer=reg),
        layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=tf.keras.losses.Huber())
    return model

modelos_logdiff      = {}
historiales_logdiff  = {}

for familia in FAMILIAS:
    cfg  = CONFIG_POR_FAMILIA[familia]
    data = datos_por_familia[familia]
    n_features = data['X_tr'].shape[2]

    model = build_model(LOOKBACK, n_features, cfg['units'], cfg['dropout'], cfg['lr'])
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)

    print(f"\nEntrenando LSTM_logdiff — Familia {familia} (units={cfg['units']}, lr={cfg['lr']})...")
    hist = model.fit(
        data['X_tr'], data['y_tr'],
        validation_data=(data['X_va'], data['y_va']),   # ← validación real, NUNCA el test
        epochs=180, batch_size=8,
        callbacks=[early_stop], verbose=0
    )
    modelos_logdiff[familia]     = model
    historiales_logdiff[familia] = hist
    print(f"  epochs entrenados: {len(hist.history['loss'])}  "
          f"best val_loss={min(hist.history['val_loss']):.4f}")


Entrenando LSTM_logdiff — Familia 106 (units=16, lr=0.0005)...
  epochs entrenados: 180  best val_loss=0.1049

Entrenando LSTM_logdiff — Familia 124 (units=32, lr=0.0003)...
  epochs entrenados: 161  best val_loss=0.0143

Entrenando LSTM_logdiff — Familia 233 (units=16, lr=0.0005)...
  epochs entrenados: 180  best val_loss=0.0605


In [7]:
# ==============================================================================
# PARTE 7: PREDICCIÓN SOBRE TEST Y REVERSIÓN DE TRANSFORMACIONES
# ==============================================================================
print("\n[Proceso] Generando predicciones y revirtiendo transformaciones...")

preds_finales = {}
true_finales  = {}
indices_test  = ventas_mensual.index[-TEST_SIZE:]

for familia in FAMILIAS:
    data  = datos_por_familia[familia]
    model = modelos_logdiff[familia]

    preds_scaled = model.predict(data['X_te'], verbose=0).flatten()
    y_te_scaled  = data['y_te']

    col_idx  = list(df_train_raw.columns).index(familia)
    n_cols   = len(df_train_raw.columns)
    dummy_p  = np.zeros((len(preds_scaled), n_cols)); dummy_p[:, col_idx] = preds_scaled
    dummy_t  = np.zeros((len(y_te_scaled), n_cols));  dummy_t[:, col_idx] = y_te_scaled

    preds_diff = scaler.inverse_transform(dummy_p)[:, col_idx]
    true_diff  = scaler.inverse_transform(dummy_t)[:, col_idx]

    preds_f = np.zeros(TEST_SIZE)
    true_f  = np.zeros(TEST_SIZE)
    for t in range(TEST_SIZE):
        fecha_actual = indices_test[t]
        idx_pax = ventas_mensual.index.get_loc(fecha_actual)
        # Usamos el valor REAL (sin winsorizar) del mes anterior para reconstruir,
        # igual que en el notebook original
        valor_real_anterior = ventas_mensual.iloc[idx_pax - 1][familia]
        log_real_anterior   = np.log1p(valor_real_anterior)
        preds_f[t] = np.expm1(log_real_anterior + preds_diff[t])
        true_f[t]  = np.expm1(log_real_anterior + true_diff[t])

    preds_finales[familia] = preds_f
    true_finales[familia]  = true_f


[Proceso] Generando predicciones y revirtiendo transformaciones...


In [8]:
# ==============================================================================
# PARTE 8 (extendida): MÉTRICAS + DIRECCIÓN DEL ERROR (ME / MPE)
# ==============================================================================
def smape(actual, forecast):
    return 100 * np.mean(2*np.abs(forecast-actual) / (np.abs(actual)+np.abs(forecast)+1e-9))

def me(actual, forecast):
    """Mean Error — con signo, en las unidades originales (kg)."""
    return np.mean(forecast - actual)

def mpe(actual, forecast):
    """Mean Percentage Error — con signo, en %."""
    return 100 * np.mean((forecast - actual) / np.where(actual == 0, 1, actual))

def calcular_metricas(actual, forecast, modelo, familia):
    mape_val = np.mean(np.abs((actual - forecast) / np.where(actual == 0, 1, actual))) * 100
    return {
        'Modelo': modelo, 'Familia': familia,
        'RMSE'   : round(np.sqrt(np.mean((actual-forecast)**2)), 2),
        'MAE'    : round(np.mean(np.abs(actual-forecast)), 2),
        'MAPE_%' : round(mape_val, 2),
        'SMAPE_%': round(smape(actual, forecast), 2),
        'ME'     : round(me(actual, forecast), 2),
        'MPE_%'  : round(mpe(actual, forecast), 2),
    }

print("\n" + "="*60)
print("       RESULTADOS LSTM_logdiff (con dirección del error)")
print("="*60)

resultados_logdiff = []
for familia in FAMILIAS:
    actual   = true_finales[familia]
    forecast = preds_finales[familia]
    m = calcular_metricas(actual, forecast, 'LSTM_logdiff', familia)
    resultados_logdiff.append(m)

    direccion = "SOBRE-estima" if m['MPE_%'] > 0 else "INFRA-estima"
    print(f"Familia {familia}:")
    print(f"  MAE={m['MAE']:,.2f}  RMSE={m['RMSE']:,.2f}  "
          f"MAPE={m['MAPE_%']:.2f}%  SMAPE={m['SMAPE_%']:.2f}%")
    print(f"  ME={m['ME']:>+10,.2f} kg   MPE={m['MPE_%']:>+7.2f}%   -> el modelo {direccion} de media")
    print("-"*60)

df_logdiff = pd.DataFrame(resultados_logdiff)
df_logdiff.to_csv('../data/processed/06_metricas_logdiff.csv', index=False)
print("\nCSV guardado: data/processed/06_metricas_logdiff.csv")


       RESULTADOS LSTM_logdiff (con dirección del error)
Familia 106:
  MAE=7,756.60  RMSE=11,168.40  MAPE=144.17%  SMAPE=117.85%
  ME= -5,850.78 kg   MPE= +46.03%   -> el modelo SOBRE-estima de media
------------------------------------------------------------
Familia 124:
  MAE=15,218.84  RMSE=18,341.69  MAPE=30.83%  SMAPE=31.96%
  ME= -2,951.62 kg   MPE=  -0.71%   -> el modelo INFRA-estima de media
------------------------------------------------------------
Familia 233:
  MAE=15,489.79  RMSE=21,254.47  MAPE=109.70%  SMAPE=53.31%
  ME= +5,204.58 kg   MPE= +81.00%   -> el modelo SOBRE-estima de media
------------------------------------------------------------

CSV guardado: data/processed/06_metricas_logdiff.csv
